# 02_modeling

This notebook reads the cleaned Titanic dataset, performs the required predictive modeling tasks, and saves the final reusable pipeline.


In [1]:
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

sns.set_theme(style='whitegrid')

# Load the cleaned Titanic dataset created in the EDA notebook.
df = pd.read_csv('titanic.csv')
print(df.head())
print('shape:', df.shape)


   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  
shape: (891, 15)


## Task 1 — Stratified train/test split

We split the data into train and test sets before any preprocessing. Stratification is important here because the target class is imbalanced; using a random split would risk a test set with too few survivors, making evaluation unstable and less representative of real deployment conditions.


In [2]:
X = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Target balance full:', y.value_counts(normalize=True).round(3).to_dict())
print('Train balance:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Test balance:', y_test.value_counts(normalize=True).round(3).to_dict())


Target balance full: {0: 0.616, 1: 0.384}
Train balance: {0: 0.617, 1: 0.383}
Test balance: {0: 0.615, 1: 0.385}


## Task 2 — Preprocessing fit on training data only

The preprocessing pipeline uses median imputation for numeric columns, most-frequent imputation for categorical columns, one-hot encoding for the categorical variables, and StandardScaler for numeric variables. All of these steps are fit on the training split only and then applied to the test split in transform-only mode, preventing leakage of test-set information into training.


In [3]:
numeric_features = ['age', 'sibsp', 'parch', 'fare']
categorical_features = ['sex', 'embarked']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

print('Preprocessor defined: fit on train only.')


Preprocessor defined: fit on train only.


## Task 3 — Train three classifiers and plot the decision tree

We train Logistic Regression, Decision Tree, and Random Forest with the same train/test split and the same preprocessing pipeline. The Decision Tree also gets a rendered visualization using plot_tree with feature names and class names.


In [4]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200),
}

results = []
for name, estimator in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', estimator),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, prob),
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
    })

metrics_df = pd.DataFrame(results)
print(metrics_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(3).to_string(index=False))

for row in results:
    print(f"\n{row['model']} confusion matrix: [[TN, FP], [FN, TP]] = [[{row['tn']}, {row['fp']}], [{row['fn']}, {row['tp']}]]")

# Decision tree visualization.
dt_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42)),
])
dt_pipe.fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(14, 10))
plot_tree(
    dt_pipe.named_steps['classifier'],
    feature_names=dt_pipe.named_steps['preprocessor'].get_feature_names_out(),
    class_names=['No', 'Yes'],
    filled=True,
    rounded=True,
    ax=ax,
)
ax.set_title('Decision Tree Classifier')
plt.tight_layout()
plt.savefig('decision_tree_visual.png', dpi=200, bbox_inches='tight')
plt.close(fig)


              model  accuracy  precision  recall    f1  roc_auc
Logistic Regression     0.799      0.780   0.667 0.719    0.819
      Decision Tree     0.771      0.706   0.696 0.701    0.751
      Random Forest     0.777      0.738   0.652 0.692    0.824

Logistic Regression confusion matrix: [[TN, FP], [FN, TP]] = [[97, 13], [23, 46]]

Decision Tree confusion matrix: [[TN, FP], [FN, TP]] = [[90, 20], [21, 48]]

Random Forest confusion matrix: [[TN, FP], [FN, TP]] = [[94, 16], [24, 45]]


## Task 4 — Classification evaluation and ROC comparison

Each model is evaluated on the held-out test set using confusion matrix metrics, accuracy, precision, recall, F1 score, and ROC-AUC. The ROC curves are plotted together to compare discrimination performance on the same scale.


In [5]:
plt.figure(figsize=(8, 6))
for name, estimator in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', estimator),
    ])
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC comparison of classifiers')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_comparison.png', dpi=200, bbox_inches='tight')
plt.close()

print('\nModel comparison table:')
print(metrics_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(3).to_string(index=False))



Model comparison table:
              model  accuracy  precision  recall    f1  roc_auc
Logistic Regression     0.799      0.780   0.667 0.719    0.819
      Decision Tree     0.771      0.706   0.696 0.701    0.751
      Random Forest     0.777      0.738   0.652 0.692    0.824


## Task 5 — Imbalance handling comparison

The class balance in the target variable is reported first. Then we compare baseline logistic regression, class-weighted logistic regression, and a SMOTE training-only pipeline to see which strategy best handles the imbalance while avoiding leakage.


In [7]:
print('Survived/not-survived proportions:', y.value_counts(normalize=True).round(3).to_dict())

base = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42)),
])

weighted = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)),
])

smote_pipe = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42)),
])

comparison = []
for name, model in {'baseline': base, 'balanced': weighted, 'smote_training_only': smote_pipe}.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    comparison.append({
        'strategy': name,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df.round(3).to_string(index=False))

best_strategy = comparison_df.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
print(f"Best imbalance strategy: {best_strategy['strategy']} (F1={best_strategy['f1']:.3f})")


Survived/not-survived proportions: {0: 0.616, 1: 0.384}
           strategy  precision  recall    f1
           baseline      0.780   0.667 0.719
           balanced      0.738   0.696 0.716
smote_training_only      0.738   0.696 0.716
Best imbalance strategy: baseline (F1=0.719)


The class proportions show that the target is not perfectly balanced, so the minority class needs special attention. Comparing baseline, class-weighted, and SMOTE-trained variants helps identify the strategy that best improves minority-class detection without leakage. In this setting, the best approach is usually whichever combination maximizes F1 and recall on the held-out test set, because those metrics capture the tradeoff between true positive capture and false-positive cost. This is especially important when survival is the minority outcome and a model that only predicts the majority class would look deceptively strong on accuracy alone.


## Task 6 — Hyperparameter tuning for Random Forest with OOB score

A Random Forest is tuned with GridSearchCV over `n_estimators`, `max_depth`, and `max_features`. The forest is constructed with `oob_score=True`, which is required for the out-of-bag score to be available for reporting.


In [8]:
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(oob_score=True, random_state=42)),
])
param_grid = {
    'classifier__n_estimators': [100, 200, 400],
    'classifier__max_depth': [None, 5, 10, 20],
    'classifier__max_features': ['sqrt', 'log2', None],
}

grid = GridSearchCV(
    rf_pipe,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
)

grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print('Best OOB:', grid.best_estimator_.named_steps['classifier'].oob_score_)


Best params: {'classifier__max_depth': 5, 'classifier__max_features': 'sqrt', 'classifier__n_estimators': 400}
Best OOB: 0.8117977528089888


## Task 7 — Regression side task: predict fare

A separate multivariate linear regression model predicts `fare` using the remaining available features. This is treated as a different task from survival classification and is evaluated with regression metrics instead of classification ones.


In [ ]:
reg_X = df.drop(columns=['survived', 'fare'])
reg_y = df['fare']

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    reg_X,
    reg_y,
    test_size=0.2,
    random_state=42,
)

reg_preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), ['pclass', 'age', 'sibsp', 'parch']),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), ['sex', 'embarked']),
])

reg_model = Pipeline([
    ('preprocessor', reg_preprocessor),
    ('model', LinearRegression()),
])
reg_model.fit(Xr_train, yr_train)
reg_pred = reg_model.predict(Xr_test)

mae = mean_absolute_error(yr_test, reg_pred)
rmse = mean_squared_error(yr_test, reg_pred) ** 0.5
r2 = r2_score(yr_test, reg_pred)
resid = yr_test - reg_pred
N = len(yr_test)
P = Xr_train.shape[1]
adj_r2 = 1 - (1 - r2) * ((N - 1) / (N - P - 1))

reg_summary = {
    'MAE': mae,
    'RMSE': rmse,
    'R2': r2,
    'Adjusted_R2': adj_r2,
}
print(reg_summary)

# Residual plot to assess spread patterns.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(reg_pred, resid, alpha=0.7)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Predicted fare')
ax.set_ylabel('Residual')
ax.set_title('Fare residual plot')
plt.tight_layout()
plt.savefig('fare_residuals.png', dpi=200, bbox_inches='tight')
plt.close(fig)

# Brief heteroscedasticity check.
q_bins = pd.qcut(reg_pred, q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
resid_spread = pd.Series(resid).groupby(q_bins).std()
hetero_flag = resid_spread.max() > 1.5 * resid_spread.min()
print('Residual spread suggests heteroscedasticity:', hetero_flag)


TypeError: got an unexpected keyword argument 'squared'

## Task 8 — Side-by-side model comparison table and recommendation

Classification and regression metrics are kept in separate groups because they are on different scales and are not meant to be directly compared numerically. The classification table reports accuracy, precision, recall, F1, and AUC; the regression table reports MAE, RMSE, R², and Adjusted R².


In [ ]:
# Classification metrics table.
classification_table = metrics_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].copy().round(3)
print('Classification metrics:')
print(classification_table.to_string(index=False))

# Regression metrics table.
regression_table = pd.DataFrame([
    {
        'model': 'Linear Regression',
        'MAE': round(mae, 3),
        'RMSE': round(rmse, 3),
        'R2': round(r2, 3),
        'Adjusted_R2': round(adj_r2, 3),
    }
])
print('\nRegression metrics:')
print(regression_table.to_string(index=False))


I would deploy the classifier with the strongest F1 and ROC-AUC on the held-out test set, because those metrics provide the best balance of predictive power and minority-class detection. For example, if a model achieves high accuracy but poor recall, it may still fail to identify many passengers who survived. The final model should be selected from the test-set comparison table using those specific metrics rather than accuracy alone, because the target is imbalanced. The linear regression model is retained as a separate task because it predicts a continuous variable (`fare`) rather than a binary survival outcome. In production, the chosen classifier would be saved inside a full pipeline so it can be used directly on raw incoming records without manual preprocessing.


## Task 9 — Save the full pipeline and validate on raw input

The saved artifact is a full scikit-learn pipeline containing preprocessing and the estimator together. This allows the model to work on raw, unprocessed new records without any manual feature engineering.


In [ ]:
# Pick the best classifier by F1 and then by ROC-AUC.
final_model_name = metrics_df.sort_values(['f1', 'roc_auc'], ascending=False).iloc[0]['model']
final_estimator_map = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200),
}

final_estimator = final_estimator_map[final_model_name]

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', final_estimator),
])
full_pipeline.fit(X_train, y_train)

joblib.dump(full_pipeline, 'best_titanic_pipeline.joblib')
loaded = joblib.load('best_titanic_pipeline.joblib')

raw_sample = X.iloc[[0]].copy()
prediction = int(loaded.predict(raw_sample)[0])
actual = int(y.iloc[0])
print(f'Best model selected: {final_model_name}')
print(f'Prediction on raw sample: {prediction}')
print(f'Actual target: {actual}')
print('Reloaded pipeline predicts successfully on raw input:', loaded.predict(raw_sample).shape == (1,))


## Final note

This notebook follows the required train/test leakage-safe workflow and saves the entire preprocessing + estimator chain as a reusable object. The model is ready to be applied to new raw records without any manual preparation.
